In [ ]:
%cd ../../

In [ ]:
import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import polars as pl
import joblib
import numpy as np
from darts.timeseries import concatenate
from darts import TimeSeries
from pytorch_lightning.callbacks import TQDMProgressBar
from lightning.pytorch.loggers import TensorBoardLogger
from sklearn.metrics import root_mean_squared_error, mean_absolute_percentage_error
from darts.models import CatBoostModel, TFTModel, TransformerModel, TSMixerModel, RNNModel, XGBModel, LightGBMModel, NBEATSModel
from darts.dataprocessing.transformers import Scaler, BoxCox, Diff
from sklearn.preprocessing import MinMaxScaler

In [ ]:
plt.style.use('seaborn-v0_8')
plt.rcParams.update({'font.size': 8})

# Load data and dims

## Load and process data

In [ ]:
EPS = 0

In [ ]:
path =  "data/processed/pos_daily_snapshot.parquet"
pos_daily_snapshot = pl.read_parquet(path)

pos_daily_snapshot.head()

In [ ]:
pos_daily_snapshot = (
    pos_daily_snapshot

    # Pivot
    .pivot(index='date', on='restaurant', values='pcs')
    .sort('date')
)

pos_daily_snapshot.head()

In [ ]:
restaurant = "1"

series = (
    TimeSeries
    .from_dataframe(
        pos_daily_snapshot.to_pandas(),
        time_col='date',
        value_cols=restaurant,
        fillna_value=EPS,
        freq='B',
    )
)

series.plot()

In [ ]:
# static_covs_multi = pd.DataFrame(data={"restaurant": [0, 1, 2, 3]})
# series = series.with_static_covariates(static_covs_multi)
# series.static_covariates

## Load dims

### `dim_exam`

In [ ]:
path = "data/processed/dim_exams_uhelsinki.xlsx"
dim_exam = (
     pl.read_excel(path)
    .filter(pl.col('date').dt.weekday() <= 5)
)


series_exam = (
    TimeSeries
    .from_dataframe(
        df=dim_exam.to_pandas(),
        time_col='date',
        freq='b',
        fill_missing_dates = False,
        value_cols='is_exam'
    )
)
series_exam.plot()

### `dim_holiday`

In [ ]:
path = "data/processed/dim_holidays_uhelsinki.xlsx"
dim_holiday = (
    pl.read_excel(path)
    .filter(pl.col('date').dt.weekday() <= 5)
)


series_holiday = (
    TimeSeries
    .from_dataframe(
        df=dim_holiday.to_pandas(),
        time_col='date',
        freq='b',
        fill_missing_dates = True,
        value_cols='is_holiday'
    )
    .astype("int")
)
series_holiday.plot()


In [ ]:
CUTOFF_DATE = pd.to_datetime("2025-01-01")
series_train, series_test = series.astype(np.float32).split_before(CUTOFF_DATE)

series_cov = concatenate(
    [
        series_exam,
        series_holiday,
    ],
    axis=1
).astype("int")



# Plot
# fig = plt.figure(figsize=(8, 6))
# ax = fig.add_subplot(111)

# Scaler(MinMaxScaler(feature_range=(0, 1))).fit_transform(series).plot(ax=ax, label='pos')
# series_cov.plot(ax=ax)

# Train

In [ ]:
# transformer = Diff(1, dropna=True)
# series_transformed = transformer.fit_transform(series)
# series_train_diff = series_transformed.drop_after(CUTOFF_DATE)


series_train_transformed = series_train
# series_train_transformed = series_train_diff

transformer_target = Scaler(MinMaxScaler(feature_range=(-1, 1)))
# transformer_target = BoxCox(lmbda=0.)
series_train_transformed = transformer_target.fit_transform(series_train_transformed)


series_train_transformed.plot()

## ML and statistical models

In [ ]:
version = datetime.datetime.now().strftime("%m-%d_%H-%M-%S")
model_name = "catboost"

input_chunk_length = 5
output_chunk_length = 1
num_epochs = 500


# Define params
add_encoders = {
    "datetime_attribute": {
        "future": ["dayofweek", 'day', 'month'],
        "past": ["dayofweek", 'day', 'month']
    },
    'cyclic': {
        'past': ["dayofweek", 'day', 'month'],
        'future': ["dayofweek", 'day', 'month']
    },
}

params_ml = {
    "lags": input_chunk_length,
    "lags_future_covariates": [0],
    "output_chunk_length": output_chunk_length,
    "add_encoders": {**add_encoders}
}
params_dl = {
    "input_chunk_length": input_chunk_length,
    "output_chunk_length": output_chunk_length,
    "add_encoders": {**add_encoders},  
    "n_epochs": num_epochs,
    "pl_trainer_kwargs": {
        "callbacks": [TQDMProgressBar(refresh_rate=4)],
        "logger": [
            TensorBoardLogger("logs/tensorboard", name=f"{model_name}-{restaurant}", version=version, default_hp_metric=False)
        ],
        'precision': "32-true",
    },
    "optimizer_kwargs": {
        'lr': 5e-4
    }
}

# Define models
match model_name:
    # =================================================
    # ML models
    # =================================================
    case "catboost":
        model = CatBoostModel(**params_ml)
        model.fit(
            series_train_transformed,
            future_covariates=series_cov,
        )
    case "xgboost":
        model = XGBModel(**params_ml)
        model.fit(
            series_train_transformed,
            future_covariates=series_cov,
        )
    case "lightbgm":
        model = LightGBMModel(**params_ml)
        model.fit(
            series_train_transformed,
            future_covariates=series_cov,
        )

    # =================================================
    # DL models
    # =================================================
    case 'rnn':
        params_dl['model'] = 'LSTM'

        model = RNNModel(**params_dl)
        model.fit(
            series_train_transformed,
            future_covariates=series_cov,
        )
    case "tsmixer":
        model = TSMixerModel(**params_dl)
        model.fit(
            series_train_transformed,
            future_covariates=series_cov,
        )
    case "transformer":
        model = TransformerModel(**params_dl)
        model.fit(
            series_train_transformed,
            past_covariates=series_cov.split_before(CUTOFF_DATE)[0],
        )
    case "tft":
        params_dl["categorical_embedding_sizes"] = {
            "holiday": (2, 2),
            "exam": (2, 2),
            "restaurant": (4, 4)
        }

        model = TFTModel(**params_dl)
        model.fit(
            series_train_transformed,
            future_covariates=series_cov,
        )
    case "n-beats":
        model = NBEATSModel(**params_dl)
        model.fit(
            series_train_transformed,
            past_covariates=series_cov.split_before(CUTOFF_DATE)[0],
        )
    
    case _:
        raise NotImplementedError()

### Test

In [ ]:
match model_name:
    # =================================================
    # ML models
    # =================================================
    case "catboost":
        series_pred = model.predict(
            n=len(series_test),
            future_covariates=series_cov
        )
    case "xgboost":
        series_pred = model.predict(
            n=len(series_test),
            future_covariates=series_cov
        )
    case "lightbgm":
        series_pred = model.predict(
            n=len(series_test),
            future_covariates=series_cov
        )

    # =================================================
    # DL models
    # =================================================
    case "rnn":
        series_pred = model.predict(
            n=len(series_test),
            future_covariates=series_cov
        )
    case "tsmixer":
        series_pred = model.predict(
            n=len(series_test),
            future_covariates=series_cov
        )
    case "transformer":
        series_pred = model.predict(
            n=len(series_test),
            past_covariates=series_cov
        )
    case "transformer":
        series_pred = model.predict(
            n=len(series_test),
            past_covariates=series_cov
        )
    case "tft":
        series_pred = model.predict(
            n=len(series_test),
            future_covariates=series_cov
        )
    case "n-beats":
        series_pred = model.predict(
            n=len(series_test),
            past_covariates=series_cov
        )

    case _:
        raise NotImplementedError()
    

series_pred = transformer_target.inverse_transform(series_pred)
# series_pred = transformer.inverse_transform(series_train_diff.concatenate(series_pred)).drop_before(CUTOFF_DATE)

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111)
series_pred.plot(ax=ax, label='pred')
series_test.plot(ax=ax, label='gt')
# rmse_val = mape(series_test[comp], series_pred[comp])

df = pd.concat(
    [
        series_test.to_dataframe().rename(columns={restaurant: "gt"}),
        series_pred.to_dataframe().rename(columns={restaurant: "pred"})
    ],
    axis=1
)
df = df[df['gt'] > EPS]
mape_val = mean_absolute_percentage_error(df['gt'], df['pred'])

ax.set_title(f"{restaurant} | mape = {mape_val*100:.4}%")

# Export to production model

Use entire `series` and `series_cov` for prediction

In [ ]:
tag = "May_26"
restaurant = "4"

path_dir = Path("trained_models/whole_restaurant_pos") / tag / restaurant

path_dir.mkdir(exist_ok=True, parents=True)

path_model = path_dir / "model.pt"

In [ ]:
series = (
    TimeSeries
    .from_dataframe(
        pos_daily_snapshot.to_pandas(),
        time_col='date',
        value_cols=restaurant,
        fillna_value=EPS,
        freq='B',
    )
    .astype(np.float32)
)

series_cov = concatenate([
    series_exam,
    series_holiday,
], axis=1).astype("int")

In [ ]:
# assert isinstance(series_cov_transformed, TimeSeries)


input_chunk_length = 5
output_chunk_length = 1
num_epochs = 20000


# Define params
add_encoders = {
    "datetime_attribute": {
        "future": ["dayofweek", 'day', 'month'],
    },
    'cyclic': {
        'future': ["dayofweek", 'day', 'month'],
    },
}

params_ml = {
    "lags": input_chunk_length,
    "lags_future_covariates": [0],
    "output_chunk_length": output_chunk_length,
    "add_encoders": {**add_encoders},
    "iterations": num_epochs,
    # "learning_rate": 1e-3
}

# model = TransformerModel(**params_dl)
# model.fit(series_transformed, past_covariates=series_cov_transformed)
model = CatBoostModel(**params_ml, categorical_future_covariates=['is_holiday', 'is_exam'])
model.fit(series, future_covariates=series_cov)

### Verify model training with backtest

In [ ]:
idx = pd.to_datetime('2024-05-01')
preds_raw = model.historical_forecasts(series, start=idx, retrain=False)
# preds = transformer_target.inverse_transform(preds_raw)
preds = preds_raw
assert isinstance(preds, TimeSeries)

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111)
preds.plot(ax=ax, label='pred')
series[idx:].plot(ax=ax, label='gt')
(series_cov * 900)[preds.time_index].plot(ax=ax, label='cov')

## Save model and scaler

In [ ]:
model.save(path_model.as_posix(), clean=False)

## Test loading model and making prediction

In [ ]:
model = CatBoostModel.load(path_model)
preds = model.predict(20)

preds.plot()